# Predicting `arrears_next_month`
Methodology follows `checklist.md`. Key decisions:
- `meter_ref` is an identifier: dropped. `billing_month` is used only to split, not as a feature.
- `final_notice_sent` is recorded in the billing cycle *after* the billing month, so it is not known at prediction time (and is likely a consequence of the outcome): dropped.
- The model is for future months, so the test set is the latest months. Accounts repeat across rows, so the test set also holds whole accounts: training uses only accounts that are not in the test account set.

In [ ]:
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
from sklearn.base import clone
df = pd.read_csv('data.csv', parse_dates=['billing_month'])
print(df.shape); print(df.groupby('billing_month').arrears_next_month.mean().round(3).tail(14))

The arrears rate rises sharply during 2025, so the target drifts; later months are not like earlier ones. This is why a time-based split matters.

## 1. Test set (set aside before any modelling)
Test = rows from 2025-10 onward for a random 20% of accounts. Training = rows up to 2025-08 for the other 80% of accounts (2025-09 is skipped as a gap, because its label looks into 2025-10).

In [ ]:
rng = np.random.default_rng(0)
accts = df.meter_ref.unique()
test_accts = set(rng.choice(accts, size=int(0.2*len(accts)), replace=False))
is_test_acct = df.meter_ref.isin(test_accts)
test = df[is_test_acct & (df.billing_month >= '2025-10-01')].copy()
train = df[~is_test_acct & (df.billing_month <= '2025-08-01')].copy().reset_index(drop=True)
assert not set(train.meter_ref) & set(test.meter_ref)
print(len(train), len(test), train.arrears_next_month.mean().round(3), test.arrears_next_month.mean().round(3))

## 2. Candidate models (named before any scoring)
Features known at prediction time: `tariff, occupants, kwh, bill_eur, late_payments_12m`.
1. **baseline**: predicts the training prevalence (`DummyClassifier`).
2. **logreg**: logistic regression (scaled, one-hot tariff), C in {0.01, 0.1, 1}.
3. **hgb**: gradient boosting, max_depth in {2, 3}, learning_rate 0.05, 150 iterations.
4. **logreg_lp**: logistic regression on `late_payments_12m` alone, a small-feature model.

Selection metric: average precision (PR-AUC) over CV folds (also report ROC-AUC). Threshold: chosen on CV out-of-fold predictions to maximise F1.

## 3. Cross-validation inside training only
Same structure as the test split: each fold validates on a later time block, using a random half of accounts; it trains only on earlier rows of the *other* accounts. All preprocessing sits inside the pipeline, so it is fitted within each fold.

In [ ]:
FEATS = ['tariff','occupants','kwh','bill_eur','late_payments_12m']
months = np.sort(train.billing_month.unique())
blocks = np.array_split(months, 8)[3:]          # last 5 blocks are validation blocks
def folds(seed=1):
    r = np.random.default_rng(seed); ta = train.meter_ref.unique()
    for b in blocks:
        va_accts = set(r.choice(ta, size=len(ta)//2, replace=False))
        in_va = train.meter_ref.isin(va_accts)
        tr = train.index[~in_va & (train.billing_month < b[0])]
        va = train.index[in_va & train.billing_month.isin(b)]
        yield tr, va
FOLDS = list(folds())
for tr,va in FOLDS: print(len(tr), len(va), train.loc[va,'arrears_next_month'].mean().round(3))

num = ['occupants','kwh','bill_eur','late_payments_12m']
def pre(cols):
    t=[('n',StandardScaler(),[c for c in cols if c in num])]
    if 'tariff' in cols: t.append(('c',OneHotEncoder(handle_unknown='ignore'),['tariff']))
    return ColumnTransformer(t)
def mk(est, cols): return Pipeline([('p',pre(cols)),('m',est)])
CANDS = {'baseline': (DummyClassifier(strategy='prior'), FEATS)}
for C_ in [0.01,0.1,1]: CANDS[f'logreg_C{C_}'] = (LogisticRegression(C=C_,max_iter=1000), FEATS)
for d in [2,3]: CANDS[f'hgb_d{d}'] = (HistGradientBoostingClassifier(max_depth=d,learning_rate=0.05,max_iter=150,random_state=0), FEATS)
CANDS['logreg_lp'] = (LogisticRegression(max_iter=1000), ['late_payments_12m'])
X, y = train, train.arrears_next_month
rows, oof = [], {}
for name,(est,cols) in CANDS.items():
    ap, auc, p_all, idx = [], [], [], []
    for tr,va in FOLDS:
        m = mk(clone(est), cols).fit(X.loc[tr,cols], y[tr])
        p = m.predict_proba(X.loc[va,cols])[:,1]
        ap.append(average_precision_score(y[va],p)); auc.append(roc_auc_score(y[va],p))
        p_all.append(p); idx.append(va)
    oof[name]=(np.concatenate(idx),np.concatenate(p_all))
    rows.append((name,np.mean(ap),np.std(ap),np.mean(auc)))
cv = pd.DataFrame(rows,columns=['model','cv_AP','AP_sd','cv_AUC']).sort_values('cv_AP',ascending=False); cv

## 4. Choice (CV only)

In [ ]:
best = cv.iloc[0].model
idx,p = oof[best]; yy = y[idx].values
ths = np.linspace(0.02,0.8,79)
f1s = [f1_score(yy,p>=t) for t in ths]
THR = float(ths[int(np.argmax(f1s))])
print('chosen model:', best, '| threshold:', round(THR,3), '| CV F1:', round(max(f1s),3))

**Decision recorded before touching the test set:** the model and threshold printed above (highest CV average precision; threshold maximising out-of-fold F1) are final. No further changes are made after test scoring.

## 5. Refit on all training rows; score the test set once

In [ ]:
est,cols = CANDS[best]
final = mk(clone(est), cols).fit(train[cols], y)
pt = final.predict_proba(test[cols])[:,1]; yt = test.arrears_next_month
base = mk(clone(CANDS['baseline'][0]), FEATS).fit(train[FEATS], y).predict_proba(test[FEATS])[:,1]
print(f'TEST  AP={average_precision_score(yt,pt):.3f}  ROC-AUC={roc_auc_score(yt,pt):.3f}  F1@thr={f1_score(yt,pt>=THR):.3f}')
print(f'prevalence (AP of a random ranker) = {yt.mean():.3f};  baseline AUC = {roc_auc_score(yt,base):.3f}')

## Conclusion
See the printed test numbers above. Interpretation caveats: arrears prevalence climbs steeply over time, so the test period has a much higher base rate than most training months. Calibration and the F1 threshold may therefore be off even if the ranking (AUC/AP) holds. The test set is only ~1,000 rows from about 275 accounts, so the estimates are noisy. `final_notice_sent` was excluded as a leak. The features were used as recorded and no `billing_month` trend feature was added.